# 01. Knowledge-base pipeline (documented walkthrough)

This notebook **is the documented orchestration** of the ICD-11 knowledge-base build
(the same flow as `components/main.py`).

## Pipeline overview

```
ICD-11 PDF  --chunker-->  icd11_chunks.json  --ingestion-->  ChromaDB vectors
```

| Step | Module | Input | Output |
|------|--------|-------|--------|
| 1. Chunking | `components/chunker.py` | `knowledge_base/icd_11.pdf` | `knowledge_base/icd11_chunks.json` |
| 2. Ingestion | `components/ingestion.py` | chunks JSON | `knowledge_base/chroma_db/` |

Heavy lifting stays in those modules (PDF parsing / embeddings). This notebook
holds the **control flow, explanation, and inspection** that used to live only
as a thin CLI in `main.py`.

CLI still works for automation: `python -m components.main`


In [1]:
# Path setup — works from repo root or notebooks/
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
if (_cwd / "components" / "config.py").exists():
    root = _cwd
elif (_cwd.parent / "components" / "config.py").exists():
    root = _cwd.parent
else:
    raise RuntimeError(
        "Could not locate project root. Open this notebook from the repo or notebooks/ folder."
    )

sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "retriever"))

from components.config import (
    PROJECT_ROOT,
    PDF_PATH,
    CHUNKS_PATH,
    CHROMA_PATH,
    RAG_EVAL_SUBSET_PATH,
    RAG_DEV_SLICE_PATH,
    RAG_EVAL_LABELS,
    RETRIEVAL_SECTIONS,
    MOOD_DISORDER_PREFIXES,
    DATASET_PATH,
)

print("Project root:", PROJECT_ROOT)
print("PDF:         ", PDF_PATH, "| exists=", PDF_PATH.exists())
print("Chunks:      ", CHUNKS_PATH, "| exists=", CHUNKS_PATH.exists())
print("ChromaDB:    ", CHROMA_PATH, "| exists=", CHROMA_PATH.exists())
print("Final eval:  ", RAG_EVAL_SUBSET_PATH, "| exists=", RAG_EVAL_SUBSET_PATH.exists())
print("Dev slice:   ", RAG_DEV_SLICE_PATH, "| exists=", RAG_DEV_SLICE_PATH.exists())
print("Labels:      ", list(RAG_EVAL_LABELS))
print("Sections:    ", RETRIEVAL_SECTIONS)


Project root: C:\Users\Ramy\AI-group-project-2026
PDF:          C:\Users\Ramy\AI-group-project-2026\knowledge_base\icd_11.pdf | exists= True
Chunks:       C:\Users\Ramy\AI-group-project-2026\knowledge_base\icd11_chunks.json | exists= True
ChromaDB:     C:\Users\Ramy\AI-group-project-2026\knowledge_base\chroma_db | exists= True
Final eval:   C:\Users\Ramy\AI-group-project-2026\datasets\rag_eval_subset.csv | exists= True
Dev slice:    C:\Users\Ramy\AI-group-project-2026\datasets\rag_dev_slice.csv | exists= True
Labels:       ['suicidal', 'depression', 'normal']
Sections:     ['Essential Features', 'Boundary with Normality']


## Configuration

Paths and knobs come from `components/config.py` so every entrypoint agrees
on the same PDF page range, chunk path, and embedding model.


In [2]:
# Paths already imported in the path-setup cell.
# Extra knobs for chunking / ingestion:
from components.config import (
    COLLECTION_NAME,
    EMBEDDING_MODEL,
    BATCH_SIZE,
    CONTENT_START_PAGE,
    CONTENT_END_PAGE,
)

print("Ready to run KB pipeline against knowledge_base/")
print("Collection: ", COLLECTION_NAME)
print("Embed model:", EMBEDDING_MODEL)
print("Batch size: ", BATCH_SIZE)
print("PDF pages:  ", CONTENT_START_PAGE, "->", CONTENT_END_PAGE)
assert PDF_PATH.exists() or True  # PDF optional if chunks already exist
assert CHUNKS_PATH.exists(), f"Missing chunks JSON at {CHUNKS_PATH} — set RUN_CHUNKING=True or provide the file"
print("Chunks OK:", CHUNKS_PATH)


Ready to run KB pipeline against knowledge_base/
Collection:  icd11_clinical
Embed model: FremyCompany/BioLORD-2023
Batch size:  64
PDF pages:   70 -> 676
Chunks OK: C:\Users\Ramy\AI-group-project-2026\knowledge_base\icd11_chunks.json


## Step 1: Chunking (PDF → JSON)

### What this step does
1. Extract text from the ICD-11 CDDR PDF for the clinical page range.
2. Parse disorder codes / section headings into structured chunk dicts.
3. Split oversized sections with word overlap.
4. Write `knowledge_base/icd11_chunks.json`.

### Why it matters for RAG
Retrieval quality depends on chunk boundaries. We keep section-aware clinical
units (e.g. Essential Features, Boundary with Normality) rather than naive
fixed-size windows.

Set `RUN_CHUNKING = True` only when the PDF is present and you intend to rebuild.


In [4]:
from components.chunker import (
    run_chunking,
    MAX_CHUNK_WORDS,
    CHUNK_WORD_OVERLAP,
)

# Inspect existing artifacts by default. Flip to True only to rebuild from PDF.
RUN_CHUNKING = True

print(f"MAX_CHUNK_WORDS={MAX_CHUNK_WORDS}, OVERLAP={CHUNK_WORD_OVERLAP}")
print(f"RUN_CHUNKING={RUN_CHUNKING}")

if RUN_CHUNKING:
    if not PDF_PATH.exists():
        raise FileNotFoundError(f"Missing PDF at {PDF_PATH}")
    print("\n=== Step 1/2: Chunking ===")
    chunks = run_chunking(
        pdf_path=str(PDF_PATH),
        chunks_path=str(CHUNKS_PATH),
        start_page=CONTENT_START_PAGE,
        end_page=CONTENT_END_PAGE,
        max_words=MAX_CHUNK_WORDS,
        overlap_words=CHUNK_WORD_OVERLAP,
    )
    print(f"Wrote {len(chunks)} chunks -> {CHUNKS_PATH}")
else:
    print("Skipping chunking (inspect mode). Using existing JSON if present.")


MAX_CHUNK_WORDS=220, OVERLAP=35
RUN_CHUNKING=True

=== Step 1/2: Chunking ===
Extracting pages 70–676 from PDF …
pdftotext     : C:\Users\Ramy\anaconda3\envs\rag_for_mental_health\Library\bin\pdftotext.EXE
PDF path      : C:\Users\Ramy\AI-group-project-2026\knowledge_base\icd_11.pdf
File exists   : True
  Extracted 1,938,727 characters

Parsing into chunks …
  Raw chunks : 1005
  After split: 1563 (max_words=220, overlap=35)
  Dropped 9 junk/appendix chunks
  Final chunks: 1554

Top disorders by chunk count:
    56  6A00.Z  Disorder of intellectual development, unspecified
    42  6A8Z  Mood disorder, unspecified
    29  6A02  Autism spectrum disorder
    27  6E21  Mental and behavioural disorders associated with pregna
    25  6C4G.3  Intoxication due to unknown or unspecified psychoactive
    24  6C4G.4  Withdrawal due to unknown or unspecified psychoactive s
    23  6D11.5  Borderline pattern
    18  6A0Z  Neurodevelopmental disorder, unspecified
    17  6B20.Z  Obsessive-compulsive

## Inspect chunks

After chunking (or if JSON already exists), inspect structure and content.
Each chunk typically includes disorder metadata plus clinical text fields used
for BM25 (`prompt_text`) and/or embedding (`embed_text` / `text`).


In [5]:
import json
from collections import Counter

if not CHUNKS_PATH.exists():
    raise FileNotFoundError(
        f"{CHUNKS_PATH} not found. Set RUN_CHUNKING=True or obtain the JSON artifact."
    )

with open(CHUNKS_PATH, encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Total chunks: {len(chunks)}")
print("Fields:", sorted(chunks[0].keys()))
print("\nTop sections:")
for name, n in Counter(c.get("section", "<none>") for c in chunks).most_common(8):
    print(f"  {n:5d}  {name}")

print("\n--- Examples ---")
for c in chunks[:3]:
    body = (c.get("prompt_text") or c.get("text") or "").replace("\n", " ")
    print(f"\n[{c.get('disorder_code')}] {c.get('disorder_name')} — {c.get('section')}")
    print(body[:320], "...")


Total chunks: 1554
Fields: ['disorder_code', 'disorder_name', 'domain', 'embed_text', 'section', 'source', 'text', 'word_count']

Top sections:
    399  Overview
    360  Differential Diagnosis
    276  Essential Features
    136  Additional Clinical Features
     98  Developmental Presentations
     87  Boundary with Normality
     83  Course Features
     64  Culture-Related Features

--- Examples ---

[6C4G.71] Anxiety disorder induced by unknown or unspecified specified — Overview
Anxiety disorder induced by unknown or unspecified specified psychoactive substances From secondary mental or behavioural syndromes associated with disorders and diseases classified elsewhere: ...

[6E63] Secondary anxiety syndrome — Overview
Secondary anxiety syndrome 54 Clinical Descriptions and Diagnostic Requirements for ICD-11 Mental, Behavioural or Neurodevelopmental Disorders Obsessive-compulsive and related disorders ...

[6B2Z] Obsessive-compulsive or related disorder, unspecified — Overview
Obse

## Step 2: Ingestion (JSON → ChromaDB)

### What this step does
1. Load chunk JSON.
2. Embed each chunk with BioLORD-2023 (`sentence-transformers`).
3. Upsert vectors into a persistent Chroma collection.

### Why BioLORD
Domain embeddings improve dense retrieval over general-purpose models for
clinical wording in ICD-11 text.

This step can take several minutes on CPU. Use `REBUILD_CHROMA=True` only when
you need a clean re-index.


In [6]:
from components.ingestion import run_ingestion

# Inspect existing Chroma DB by default. Flip to True to re-embed.
RUN_INGESTION = True
REBUILD_CHROMA = True

print(f"RUN_INGESTION={RUN_INGESTION}, REBUILD_CHROMA={REBUILD_CHROMA}")

if RUN_INGESTION:
    if not CHUNKS_PATH.exists():
        raise FileNotFoundError(f"Missing chunks at {CHUNKS_PATH}")
    print("\n=== Step 2/2: Ingestion ===")
    run_ingestion(
        chunks_path=str(CHUNKS_PATH),
        chroma_path=str(CHROMA_PATH),
        collection_name=COLLECTION_NAME,
        embedding_model_name=EMBEDDING_MODEL,
        batch_size=BATCH_SIZE,
        rebuild=REBUILD_CHROMA,
    )
    print("Ingestion finished.")
else:
    print("Skipping ingestion (inspect mode).")
    print("Chroma path exists=", CHROMA_PATH.exists())


RUN_INGESTION=True, REBUILD_CHROMA=True

=== Step 2/2: Ingestion ===
GPU : NVIDIA GeForce GTX 1650
VRAM: 4.29 GB

Loading: FremyCompany/BioLORD-2023 …
Loaded on  : cuda:0
Output dims: 768

ChromaDB path: C:\Users\Ramy\AI-group-project-2026\knowledge_base\chroma_db
Collection 'icd11_clinical' already exists with 1554 vectors.
Deleted existing collection (rebuild=True).
Collection ready: 'icd11_clinical'  (0 vectors)
Loaded 1554 chunks from C:\Users\Ramy\AI-group-project-2026\knowledge_base\icd11_chunks.json

Ingesting 1554 chunks into 'icd11_clinical' …



Batches: 100%|██████████| 25/25 [00:50<00:00,  2.00s/it]


Done. Collection 'icd11_clinical' now has 1554 vectors.
Ingestion finished.


## Full orchestration (equivalent to `components/main.py`)

The CLI `main()` is literally: optional chunking → ingestion → done.
The next cell mirrors that control flow so the thesis walkthrough stays in one place.


In [7]:
# Mirrors components.main:main() — flip flags above, then run this cell.
SKIP_CHUNKING = not RUN_CHUNKING
# (Chunking / ingestion already executed in the cells above when flags are True.)

print("Orchestration summary")
print(f"  skip_chunking = {SKIP_CHUNKING}")
print(f"  ran_ingestion = {RUN_INGESTION}")
print(f"  rebuild       = {REBUILD_CHROMA}")
print("\nCLI equivalents:")
print("  python -m components.main")
print("  python -m components.main --skip-chunking")
print("  python -m components.main --rebuild")
print("\nNext: notebooks/02_dataset_prep_demo.ipynb")


Orchestration summary
  skip_chunking = False
  ran_ingestion = True
  rebuild       = True

CLI equivalents:
  python -m components.main
  python -m components.main --skip-chunking
  python -m components.main --rebuild

Next: notebooks/02_dataset_prep_demo.ipynb
